# Agent 07 - Validate OHLCV 1m (Realtime)

Auditor?a estricta de `D:\\ohlcv_1m` con salida de eventos, retry queue y reporte anal?tico granular.

- Script de auditor?a: `045_agent_validate_ohlcv_1m_strict.py`
- Script de reporte: `046_agent_validate_ohlcv_1m_report.py`


In [ ]:
from pathlib import Path
import pandas as pd

SCRIPT = Path(r"C:\\TSIS_Data\\v1\\backtest_SmallCaps\\notebooks\\cell_code\\045_agent_validate_ohlcv_1m_strict.py")
DATA_ROOT = Path(r"D:\\ohlcv_1m")

RUN_ID = globals().get("RUN_ID", "20260310_ohlcv_1m_session_01")
RUN_BASE = Path(r"C:\\TSIS_Data\\v1\\backtest_SmallCaps\\runs\\ohlcv_1m_audit")
RUN_DIR = RUN_BASE / RUN_ID

MAX_FILES = int(globals().get("MAX_FILES", 99999999))
RESET_STATE = bool(globals().get("RESET_STATE", True))
RESCAN_ALL = bool(globals().get("RESCAN_ALL", True))
DISCOVERY_MODE = globals().get("DISCOVERY_MODE", "full")
RETRY_POLICY = globals().get("RETRY_POLICY", "hard_only")
MAX_RETRY_ATTEMPTS = int(globals().get("MAX_RETRY_ATTEMPTS", 3))
MAX_TS_DATE_MISMATCH_PCT = float(globals().get("MAX_TS_DATE_MISMATCH_PCT", 0.0))
MAX_TSUTC_T_MISMATCH_PCT = float(globals().get("MAX_TSUTC_T_MISMATCH_PCT", 0.0))
MAX_ZERO_VOLUME_RATIO_WARN = float(globals().get("MAX_ZERO_VOLUME_RATIO_WARN", 95.0))
CLOSEABLE_SOFT_FAIL_CAUSES = set(globals().get("CLOSEABLE_SOFT_FAIL_CAUSES", []))

if not SCRIPT.exists():
    raise FileNotFoundError(f"No existe script: {SCRIPT}")
if not DATA_ROOT.exists():
    raise FileNotFoundError(f"No existe DATA_ROOT: {DATA_ROOT}")

RUN_DIR.mkdir(parents=True, exist_ok=True)
print("RUN_ID:", RUN_ID)
print("DATA_ROOT:", DATA_ROOT)
print("RUN_DIR:", RUN_DIR)

exec(SCRIPT.read_text(encoding="utf-8"), globals(), globals())

EVENTS_CURRENT_CSV = RUN_DIR / "ohlcv_1m_events_current.csv"
RETRY_CURRENT_CSV = RUN_DIR / "retry_queue_ohlcv_1m_current.csv"
LIVE_STATUS_JSON = RUN_DIR / "live_status_ohlcv_1m.json"

print("\nArtifacts clave:")
print("-", EVENTS_CURRENT_CSV)
print("-", RETRY_CURRENT_CSV)
print("-", LIVE_STATUS_JSON)

if EVENTS_CURRENT_CSV.exists():
    ev = pd.read_csv(EVENTS_CURRENT_CSV)
    if not ev.empty and "severity" in ev.columns:
        print("\nSeverity current:")
        print(ev.groupby("severity", dropna=False).size().reset_index(name="count").to_string(index=False))

if RETRY_CURRENT_CSV.exists():
    rq = pd.read_csv(RETRY_CURRENT_CSV)
    n = rq["file"].nunique() if ("file" in rq.columns and not rq.empty) else len(rq)
    print(f"\nRetry pending files: {n}")


In [ ]:
import runpy, sys

sys.argv = [
    "046_agent_validate_ohlcv_1m_report.py",
    "--run-dir",
    str(RUN_DIR),
    # "--no-plots",
]
_ = runpy.run_path(
    r"C:\\TSIS_Data\\v1\\backtest_SmallCaps\\notebooks\\cell_code\\046_agent_validate_ohlcv_1m_report.py",
    run_name="__main__",
)


## Lectura r?pida

- `overall=OK`: sin fallos duros y sin cola retry.
- `SOFT_FAIL`: revisar causas en `TOP ROOT CAUSES`.
- `FAIL CONCENTRATION BY TICKER`: localiza tickers estructuralmente problem?ticos.
- `TEMPORAL QUALITY BY YEAR-MONTH`: detecta ventanas temporales an?malas.


In [ ]:
PS C:\Users\AlexJ> python -c "from pathlib import Path; import pandas as pd; SCRIPT=Path(r'C:\TSIS_Data\v1\backtest_SmallCaps\notebooks\cell_code\045_agent_validate_ohlcv_1m_strict.py');  DATA_ROOT=Path(r'D:\ohlcv_1m'); RUN_ID='20260310_ohlcv_1m_session_01'; RUN_DIR=Path(r'C:\TSIS_Data\v1\backtest_SmallCaps\runs\ohlcv_1m_audit')/RUN_ID; MAX_FILES=99999999; RESET_STATE=True;  RESCAN_ALL=True; DISCOVERY_MODE='full'; RETRY_POLICY='hard_only'; MAX_RETRY_ATTEMPTS=3;  MAX_TS_DATE_MISMATCH_PCT=0.0; MAX_TSUTC_T_MISMATCH_PCT=0.0; MAX_ZERO_VOLUME_RATIO_WARN=95.0;  CLOSEABLE_SOFT_FAIL_CAUSES=set(); RUN_DIR.mkdir(parents=True, exist_ok=True); print('RUN_DIR:', RUN_DIR);  exec(SCRIPT.read_text(encoding='utf-8'), globals(), globals())"


RUN_DIR: C:\TSIS_Data\v1\backtest_SmallCaps\runs\ohlcv_1m_audit\20260310_ohlcv_1m_session_01

In [1]:
from pathlib import Path
import pandas as pd

p = Path(r"D:\ohlcv_1m\ticker=AAA\year=2005\month=01\minute_aggs_AAA_2005_01.parquet")

if not p.exists():
    raise FileNotFoundError(p)

df = pd.read_parquet(p)

print("Path:", p)
print("Rows:", len(df))
print("Columns:", list(df.columns))
display(df.head(3))
display(df.tail(3))

Path: D:\ohlcv_1m\ticker=AAA\year=2005\month=01\minute_aggs_AAA_2005_01.parquet
Rows: 221
Columns: ['ticker', 'ts_utc', 'date', 'year', 'month', 'o', 'h', 'l', 'c', 'v', 'vw', 'n', 't']


,ticker,ts_utc,date,year,month,o,h,l,c,v,vw,n,t
0,AAA,2005-01-03T14:32:00Z,2005-01-03,2005,1,62.80,62.80,62.80,62.80,100.0,62.8000,1,1104762720000
1,AAA,2005-01-03T15:49:00Z,2005-01-03,2005,1,62.85,62.86,62.85,62.86,600.0,62.8567,2,1104767340000
2,AAA,2005-01-03T15:50:00Z,2005-01-03,2005,1,62.88,62.89,62.88,62.89,200.0,62.8850,2,1104767400000


,ticker,ts_utc,date,year,month,o,h,l,c,v,vw,n,t
218,AAA,2005-01-31T20:18:00Z,2005-01-31,2005,1,58.84,58.84,58.84,58.84,100.0,58.84,1,1107202680000
219,AAA,2005-01-31T20:57:00Z,2005-01-31,2005,1,58.75,58.75,58.75,58.75,400.0,58.75,1,1107205020000
220,AAA,2005-01-31T21:50:00Z,2005-01-31,2005,1,58.75,58.75,58.75,58.75,300.0,58.75,1,1107208200000


In [1]:
from pathlib import Path

SCRIPT = Path(r"C:\TSIS_Data\v1\backtest_SmallCaps\notebooks\cell_code\plot_ohlcv_1m_day_sessions.py")

if "PARQUET_PATH" in globals():
    del PARQUET_PATH

DATA_ROOT = Path(r"D:\ohlcv_1m")
CHART_STYLE = "ohlc"
SHOW_VOLUME = True

exec(SCRIPT.read_text(encoding="utf-8"), globals(), globals())

Output()

Output()

In [2]:
from pathlib import Path

SCRIPT = Path(r"C:\TSIS_Data\v1\backtest_SmallCaps\notebooks\cell_code\plot_ohlcv_1m_day_sessions.py")

if "PARQUET_PATH" in globals():
    del PARQUET_PATH

DATA_ROOT = Path(r"D:\ohlcv_1m")
CHART_STYLE = "candlestick"   # candlestick | ohlc | line
SHOW_VOLUME = True
SHOW_VWAP = False

exec(SCRIPT.read_text(encoding="utf-8"), globals(), globals())

Output()

Output()

**audit_ohlcv_1m_hours.py**

Qué hace:

- recorre D:\ohlcv_1m,
- lee cada parquet mensual 1m,
- convierte ts_utc a America/New_York,
- calcula por ticker/day:
    - primera hora observada,
    - última hora observada,
    - si hay premarket,
    - si hay RTH,
    - si hay postmarket,
    - si hay minutos fuera de política,
    - filas por bloque horario,
- usa el calendario oficial XNYS para distinguir early close,
- genera resumen por día y por ticker.

Artefactos:

```
- summary.json
- ticker_day_hours.csv
- ticker_hours_summary.csv
- status_counts.csv
- ticker_day_hours.parquet
- ticker_hours_summary.parquet
```

Por defecto los deja en:

- C:\TSIS_Data\v1\backtest_SmallCaps\runs\debug\audit_ohlcv_1m_hours

In [ ]:
from pathlib import Path

SCRIPT = Path(r"C:\TSIS_Data\v1\backtest_SmallCaps\notebooks\cell_code\audit_ohlcv_1m_hours.py")

DATA_ROOT = Path(r"D:\ohlcv_1m")
CALENDAR_CSV = Path(r"C:\TSIS_Data\v1\backtest_SmallCaps\data\reference\market_calendar_official_XNYS_20050101_20251231.csv")
AUDIT_DIR = Path(r"C:\TSIS_Data\v1\backtest_SmallCaps\runs\debug\audit_ohlcv_1m_hours")

# opcional para smoke test
MAX_TICKERS = 5

exec(SCRIPT.read_text(encoding="utf-8"), globals(), globals())